In [ ]:
print('=== INDEXACION ===\n')

docs_raw = cargar_documentos(MANIFEST, CORPUS_DIR)
chunks   = segmentar(docs_raw)

import os
import shutil
from langchain.vectorstores import Chroma


db_needs_recreation = False
if os.path.exists(DB_DIR):
    try:
        vs = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)
        if vs._collection.count() == 0:
            print(f"Chroma DB at {DB_DIR} exists but is empty. Recreating...")
            db_needs_recreation = True
    except Exception as e:
        print(f"Error loading Chroma DB from {DB_DIR}: {e}. Recreating...")
        db_needs_recreation = True
else:
    print(f"Chroma DB directory {DB_DIR} not found. Creating new DB...")
    db_needs_recreation = True

if db_needs_recreation:
    # Ensure directory is truly empty if it exists before trying to create.
    if os.path.exists(DB_DIR):
        shutil.rmtree(DB_DIR) # Remove existing (potentially corrupted) directory
    os.makedirs(DB_DIR, exist_ok=True) # Ensure directory exists for new DB
    vs = Chroma.from_documents(
        chunks,
        embedding=embeddings,
        persist_directory=DB_DIR
    )
    vs.persist()
    print(f"Chroma DB successfully created/recreated with {vs._collection.count()} documents.")
else:
    print(f"Chroma DB loaded successfully from {DB_DIR} with {vs._collection.count()} documents.")

# Verificación
test_docs = vs.similarity_search('deep learning protein structure', k=2)
for d in test_docs:
    m = d.metadata
    print(f'  [{m["doc_id"]}] p.{m.get("page",0)+1}  {d.page_content[:80]}...')
